# Transformação Silver - Custos

### Descrição
Este notebook realiza a transformação de dados da camada **Bronze** para **Silver** da tabela `custos` (origem) para `vcredit_custos` (destino).

### Adaptação de Schema
Como a ingestão Bronze foi realizada sem cabeçalho, as colunas originais (`_c0`, `_c1`, `_c2`) serão renomeadas para nomes de negócio (`id_custo`, `id_chamado`, `custo`) logo no início do processamento.

### Objetivos
* Renomear colunas técnicas para nomes de negócio
* Padronizar a coluna `custo` (remover texto "reais" e corrigir separadores)
* Converter tipo de dados de String para Decimal
* Garantir unicidade pela chave primária
* Salvar na camada Silver em formato Delta Lake

In [0]:
%sql
USE CATALOG medalhao_credit;
USE SCHEMA silver_credit;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

# Configuração de caminhos
catalogo = "medalhao_credit"
bronze_db = "bronze_credit"
silver_db = "silver_credit"

# Nome da tabela criada pela equipe (sem o prefixo vcredit_)
tabela_origem = "custos" 

print(f"🚀 Carregando tabela Bronze: {catalogo}.{bronze_db}.{tabela_origem}")

# 1. Carregar a tabela bruta
df_bronze_raw = spark.read.table(f"{catalogo}.{bronze_db}.{tabela_origem}")

# 2. Renomear colunas genéricas para nomes de negócio
# _c0 -> id_custo
# _c1 -> id_chamado
# _c2 -> custo
df_bronze = df_bronze_raw \
    .withColumnRenamed("_c0", "id_custo") \
    .withColumnRenamed("_c1", "id_chamado") \
    .withColumnRenamed("_c2", "custo")

display(df_bronze.limit(5))

### Análise Exploratória
Agora com as colunas renomeadas, analisamos o schema e uma amostra dos dados para confirmar os padrões de sujeira na coluna de valor.

In [0]:
print(f"Total de registros: {df_bronze.count()}")

# Schema e amostra
df_bronze.printSchema()

# Verificando padrões na coluna 'custo' (ex: '0.0026reais' vs '0,1816')
print("Amostra da coluna 'custo' (dados brutos):")
display(df_bronze.select("custo").sample(withReplacement=False, fraction=0.1).limit(10))

### Problemas Identificados
1.  **Formato Inconsistente:** A coluna `custo` mistura formatos. Alguns registros possuem o sufixo "reais" e ponto, outros usam vírgula como decimal.
2.  **Tipo Incorreto:** Dados numéricos estão tipados como `string`.
3.  **Nomenclatura:** O nome `custo` é genérico, vamos alterar para `valor_custo`.

---
### Transformações e Limpeza
Aplicamos a limpeza agressiva (Regex) para remover textos e padronizar o formato numérico, além de ajustar a tipagem.

In [0]:
# Tratamento da coluna de valor
df_step1 = df_bronze.select(
    F.col("id_custo"),
    F.col("id_chamado"),
    
    # Remove tudo que não for número/ponto/vírgula, troca ',' por '.' e converte
    F.regexp_replace(
        F.regexp_replace(F.col("custo"), "[^0-9,.]", ""), 
        ",", "."
    ).cast(DecimalType(18, 6)).alias("valor_custo"),
    
    # Mantendo a coluna de controle da equipe
    F.col("data_ingestao").alias("ingestion_timestamp")
)

print("Amostra após limpeza:")
display(df_step1.limit(10))
df_step1.printSchema()

### Tratamento de Duplicatas e Ordenação
Garantia de integridade da chave primária.

In [0]:
# --- Passo 2: Tratamento de Duplicatas ---
# Contagem antes
total_antes = df_step1.count()

# Remover duplicatas pelo ID (Chave Primária)
# Criamos o df_silver final a partir daqui
df_silver = df_step1.dropDuplicates(["id_custo"])

# Contagem depois
total_depois = df_silver.count()

print(f"Registros antes: {total_antes}")
print(f"Registros depois: {total_depois}")
print(f"Duplicatas removidas: {total_antes - total_depois}")

# --- Passo 3: Ordenação ---
df_silver = df_silver.orderBy(F.col("id_custo"))

display(df_silver.limit(10))

### Análise Completa de Qualidade - Custos
Agora que os dados estão limpos, calculamos métricas de qualidade e estatísticas financeiras para garantir que não perdemos dados importantes e que os valores fazem sentido.

In [0]:
# Análise completa de qualidade e estatísticas financeiras
qualidade_dados = df_silver.select([
    F.count("*").alias("total_registros"),
    F.countDistinct("id_custo").alias("id_custo_unicos"),
    F.countDistinct("id_chamado").alias("id_chamado_unicos"),
    F.avg("valor_custo").alias("custo_medio"),
    F.sum("valor_custo").alias("custo_total"),
    F.min("valor_custo").alias("custo_minimo"),
    F.max("valor_custo").alias("custo_maximo")
]).collect()[0]

print("RELATÓRIO DE QUALIDADE:")
print(f"Total de registros: {qualidade_dados['total_registros']}")
print(f"IDs custo únicos: {qualidade_dados['id_custo_unicos']}")
print(f"IDs chamado únicos: {qualidade_dados['id_chamado_unicos']}")
print("-" * 30)
print("ESTATÍSTICAS FINANCEIRAS:")
print(f"Custo Médio: R$ {qualidade_dados['custo_medio']:.2f}")
print(f"Custo Mínimo: R$ {qualidade_dados['custo_minimo']:.2f}")
print(f"Custo Máximo: R$ {qualidade_dados['custo_maximo']:.2f}")
print(f"Investimento Total Monitorado: R$ {qualidade_dados['custo_total']:.2f}")

# Verificar integridade 
nulos = df_silver.filter(F.col("id_chamado").isNull()).count()
print("-" * 30)
print(f"Registros órfãos (sem id_chamado): {nulos}")

In [0]:
%sql
DROP TABLE IF EXISTS medalhao_credit.silver_credit.vcredit_custos;

In [0]:
# Salvando na silver
tabela_destino = f"{catalogo}.{silver_db}.custos"

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(tabela_destino)

print(f"✅ Tabela {tabela_destino} salva com sucesso!")

In [0]:
display(df_silver.limit(5))